# Neural Algorithmic Reasoning for Graph-Based Maze Navigation
### Benchmarking Classical Search (BFS, Dijkstra, A*) vs Deep Graph Architectures (ResGCN, Gated MPNN)

**Project Goal:**  
Empirically compare classical graph search algorithms against deep Neural Algorithmic Reasoning (NAR) architectures—specifically **Deep Residual Graph Convolutional Networks (ResGCN)** and **Recurrent Gated Message Passing Neural Networks (GateMPNN)**—when solving shortest-path maze navigation problems.

---
### Key Methodological Features & Standards:
1. **Zero Test-Data Leakage**: Classification thresholds ($\tau^*$) are selected strictly on the validation dataset to maximize validation F1-score (`find_optimal_threshold`), and then applied as fixed thresholds across all test, unseen, and scale generalization datasets.
2. **Imbalanced Class Evaluation**: Evaluates Precision-Recall (PR-AUC / Average Precision), Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices to account for the ~75% negative class imbalance.
3. **Diameter-Matched Message Passing**: Uses 30 residual layers (ResGCN) and 30 recurrent message passing steps (GateMPNN) matching the graph diameter (~100-120 vertices).
4. **Randomized Boundary Entrances**: Automated detection of Start and Goal entrance openings across outer maze boundaries using OpenCV.
5. **Optimizer & Loss Function**: Uses **AdamW** optimizer ($lr=10^{-3}$, $weight\_decay=10^{-4}$) and **Focal BCE Loss** ($\gamma = 2.0$) to handle class imbalance.

In [1]:
import os
import random
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GCNConv, MessagePassing
from torch_geometric.utils import to_undirected
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)

# Set seeds for complete reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")

PyTorch Version: 2.5.1+cu121
Execution Device: cuda


## 1. Dataset Preparation & Graph Construction
Loads or generates 2D binary grid mazes with randomized entrance/exit locations and converts them into 4-connected NetworkX and PyTorch Geometric grid graphs.

In [2]:
from dataset_generator import prepare_kaggle_maze_dataset
from src.dataset import create_maze_dataset_from_dir
from src.graph_builder import build_maze_graph

project_root = os.getcwd()
kaggle_dir = os.path.join(project_root, "maze")
data_base = os.path.join(project_root, "data")

print("--- Step 1: Loading & Preparing Maze Datasets ---")
prepare_kaggle_maze_dataset(kaggle_dir, data_base, num_train=30, num_val=10, num_test=10, num_unseen=10)

train_dir = os.path.join(data_base, "train")
val_dir = os.path.join(data_base, "val")
test_dir = os.path.join(data_base, "test")
unseen_dir = os.path.join(data_base, "unseen")
large_dir = os.path.join(data_base, "large")
very_large_dir = os.path.join(data_base, "very_large")

train_ds = create_maze_dataset_from_dir(train_dir)
val_ds = create_maze_dataset_from_dir(val_dir)
test_ds = create_maze_dataset_from_dir(test_dir)
unseen_ds = create_maze_dataset_from_dir(unseen_dir)
large_ds = create_maze_dataset_from_dir(large_dir)
very_large_ds = create_maze_dataset_from_dir(very_large_dir)

print(f"Loaded PyG Datasets -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

--- Step 1: Loading & Preparing Maze Datasets ---
Found 1009824 Kaggle maze PNG files in 'C:\Users\a4969\nar_maze_solver\maze'. Splitting into train/val/test/unseen...
Dataset preparation complete!
Loaded PyG Datasets -> Train: 30, Val: 10, Test: 10


### Graph Topological Properties (Pandas DataFrame)
Examines test graph node counts, edge counts, graph density, and shortest path node length.

In [3]:
from src.evaluator import build_graph_info_dataframe

graph_info_df = build_graph_info_dataframe(test_ds)
display(graph_info_df.head(10))

,Maze_ID,Image_Path,Grid_Shape,Num_Nodes,Num_Edges,Start_Coords,Goal_Coords,Path_Length,BFS_Time_ms,Dijkstra_Time_ms,AStar_Time_ms
0,0,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",891,4.157,14.373,15.872
1,1,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",911,11.644,24.481,24.522
2,2,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",715,3.439,14.365,15.290
3,3,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",787,3.238,10.914,9.552
4,4,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",519,3.855,9.585,17.830
5,5,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",935,4.661,18.620,23.917
6,6,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",1195,4.633,30.348,27.297
7,7,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",891,3.995,10.277,10.639
8,8,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",883,6.249,20.424,22.766
9,9,C:\Users\a4969\nar_maze_solver\data\test\00000...,91x91,4051,8100,"(89, 90)","(1, 0)",1219,5.543,23.173,32.697


## 2. Deep GNN Architectures (ResGCN & Recurrent GateMPNN)
- **Deep ResGCN (30 Layers)**: Residual GCN blocks with LayerNorm, GELU, and initial state shortcuts ($0.1 h_0$).
- **Recurrent GateMPNN (30 Steps)**: Recurrent MessagePassing module with GRU cell update and initial feature injection.

In [4]:
from src.models import GCNMazeSolver, MPNNMazeSolver

gcn_model = GCNMazeSolver(in_channels=8, hidden_channels=64, num_layers=30, dropout=0.1).to(device)
mpnn_model = MPNNMazeSolver(in_channels=8, hidden_channels=64, num_steps=30, dropout=0.1).to(device)

print("ResGCN Model Structure:")
print(gcn_model)

print("\nRecurrent GateMPNN Model Structure:")
print(mpnn_model)

ResGCN Model Structure:
GCNMazeSolver(
  (embedding): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=64, out_features=64, bias=True)
  )
  (blocks): ModuleList(
    (0-29): 30 x ResGCNBlock(
      (conv): GCNConv(64, 64)
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

Recurrent GateMPNN Model Structure:
MPNNMazeSolver(
  (embedding): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=64, out_features=

## 3. Training Models with AdamW & Focal Loss
Trains both models using **AdamW** optimizer ($lr=1e-3, weight\_decay=1e-4$), **Focal BCE Loss** ($\gamma=2.0$), and early stopping ($patience=10$).

In [5]:
from src.trainer import train_model

print("--- Training Deep ResGCN (30 Layers) ---")
gcn_history = train_model(gcn_model, train_ds, val_ds, model_name="GCN", epochs=40, lr=1e-3, patience=10, seed=42)

print("\n--- Training Recurrent GateMPNN (30 Steps) ---")
mpnn_history = train_model(mpnn_model, train_ds, val_ds, model_name="MPNN", epochs=40, lr=1e-3, patience=10, seed=42)

--- Training Deep ResGCN (30 Layers) ---
Epoch 001/040 | Train Loss: 0.1860 | Val Loss: 0.1372 | Train F1: 0.6023 | Val F1: 0.7182
Epoch 002/040 | Train Loss: 0.1124 | Val Loss: 0.0751 | Train F1: 0.7508 | Val F1: 0.8337
Epoch 003/040 | Train Loss: 0.1179 | Val Loss: 0.3930 | Train F1: 0.7677 | Val F1: 0.1748
Epoch 004/040 | Train Loss: 0.1360 | Val Loss: 0.1814 | Train F1: 0.7539 | Val F1: 0.7009
Epoch 005/040 | Train Loss: 0.1529 | Val Loss: 0.0867 | Train F1: 0.7190 | Val F1: 0.7940
Epoch 006/040 | Train Loss: 0.0971 | Val Loss: 0.0753 | Train F1: 0.8018 | Val F1: 0.8369
Epoch 007/040 | Train Loss: 0.0869 | Val Loss: 0.0900 | Train F1: 0.8177 | Val F1: 0.8603
Epoch 008/040 | Train Loss: 0.0834 | Val Loss: 0.0566 | Train F1: 0.8398 | Val F1: 0.8782
Epoch 009/040 | Train Loss: 0.0791 | Val Loss: 0.0526 | Train F1: 0.8579 | Val F1: 0.8821
Epoch 010/040 | Train Loss: 0.0747 | Val Loss: 0.2422 | Train F1: 0.8709 | Val F1: 0.7800
Epoch 011/040 | Train Loss: 0.0620 | Val Loss: 0.0571 | Tra

## 4. Zero Test-Data Leakage: Validation Threshold Tuning & Test Evaluation
Sweeps candidate thresholds $\tau \in [0.05, 0.95]$ on `val_ds` to find $\tau^*$ maximizing Validation F1. Locks $\tau^*$ for test evaluation.

In [6]:
from src.evaluator import (
    find_optimal_threshold, evaluate_model_performance,
    benchmark_all_classical_baselines, build_comparison_dataframe
)

# 1. Benchmark Classical Search Baselines (BFS, Dijkstra, A*)
classical_summary = benchmark_all_classical_baselines(test_ds)

# 2. Find Validation Selected Thresholds (Zero Leakage)
gcn_tau = find_optimal_threshold(gcn_model, val_ds, device)
mpnn_tau = find_optimal_threshold(mpnn_model, val_ds, device)

print(f"Validation Selected Thresholds -> ResGCN tau*: {gcn_tau:.3f}, GateMPNN tau*: {mpnn_tau:.3f}")

# 3. Evaluate Models on Test Set using Fixed Validation Thresholds
gcn_test = evaluate_model_performance(gcn_model, test_ds, device, threshold=gcn_tau)
mpnn_test = evaluate_model_performance(mpnn_model, test_ds, device, threshold=mpnn_tau)

# 4. Build & Display Pandas Comparison DataFrame
comparison_df = build_comparison_dataframe(classical_summary, gcn_test, mpnn_test)
display(comparison_df)

[Validation Threshold Tuner] Selected optimal threshold tau = 0.480 (Val F1 = 0.9351)
[Validation Threshold Tuner] Selected optimal threshold tau = 0.840 (Val F1 = 0.9977)
Validation Selected Thresholds -> ResGCN tau*: 0.480, GateMPNN tau*: 0.840


,Method,Accuracy (%),Precision (%),Recall (%),F1-Score,ROC-AUC (%),Average Precision (%),Inference Time (ms),Memory Usage (MB),Computational Efficiency (nodes/ms)
0,BFS,100.00,100.00,100.00,1.0000,100.00,100.00,5.14,0.12,787.9
1,Dijkstra,100.00,100.00,100.00,1.0000,100.00,100.00,17.66,0.62,229.4
2,A*,100.00,100.00,100.00,1.0000,100.00,100.00,20.04,0.41,202.2
3,GCN (AdamW),96.98,88.94,98.58,0.9351,99.08,95.72,24.24,3.87,167.1
4,MPNN (AdamW),99.95,99.94,99.84,0.9989,100.00,100.00,14.97,3.86,270.7


## 5. Visualizing Publication Figures & Solution Overlays
Generates ROC & PR curves, confusion matrices, latency/memory profiling, scale generalization, and maze prediction overlays.

In [7]:
from src.visualizer import (
    plot_training_curves, plot_roc_and_pr_curves, plot_confusion_matrices,
    plot_performance_comparison, plot_generalization, plot_maze_prediction_overlay
)

# 1. Training and Validation Loss & F1 Curves
plot_training_curves(gcn_history, mpnn_history)

# 2. Combined ROC and Precision-Recall Curves
plot_roc_and_pr_curves({"GCN": gcn_test, "MPNN": mpnn_test})

# 3. Confusion Matrices under Validation-Tuned Thresholds
plot_confusion_matrices({"GCN": gcn_test, "MPNN": mpnn_test})

# 4. Inference Time, Memory & Throughput Comparison
plot_performance_comparison(classical_summary, gcn_test, mpnn_test)

# 5. Scale Generalization Across Unseen, Large (31x31), and Very Large (41x41) Mazes
gcn_unseen = evaluate_model_performance(gcn_model, unseen_ds, device, threshold=gcn_tau)
gcn_large = evaluate_model_performance(gcn_model, large_ds, device, threshold=gcn_tau)
gcn_very_large = evaluate_model_performance(gcn_model, very_large_ds, device, threshold=gcn_tau)

mpnn_unseen = evaluate_model_performance(mpnn_model, unseen_ds, device, threshold=mpnn_tau)
mpnn_large = evaluate_model_performance(mpnn_model, large_ds, device, threshold=mpnn_tau)
mpnn_very_large = evaluate_model_performance(mpnn_model, very_large_ds, device, threshold=mpnn_tau)

scales = ["Standard 21x21", "Unseen 21x21", "Large 31x31", "Very Large 41x41"]
gcn_f1s = [gcn_test["f1"], gcn_unseen["f1"], gcn_large["f1"], gcn_very_large["f1"]]
mpnn_f1s = [mpnn_test["f1"], mpnn_unseen["f1"], mpnn_large["f1"], mpnn_very_large["f1"]]

plot_generalization(scales, gcn_f1s, mpnn_f1s)

# 6. Visual Maze Prediction Overlay & Probability Heatmaps
sample_meta = test_ds.get_metadata(0)
sample_data = test_ds[0].to(device)

with torch.no_grad():
    gcn_logits = gcn_model(sample_data.x, sample_data.edge_index).squeeze(-1)
    mpnn_logits = mpnn_model(sample_data.x, sample_data.edge_index).squeeze(-1)
    gcn_probs = torch.sigmoid(gcn_logits).cpu().numpy()
    mpnn_probs = torch.sigmoid(mpnn_logits).cpu().numpy()

G, node_to_idx, _ = build_maze_graph(
    sample_meta["binary_grid"], sample_meta["start_coords"], sample_meta["goal_coords"]
)

plot_maze_prediction_overlay(sample_meta, gcn_probs, mpnn_probs, node_to_idx)